# Revisione delle credenze su un frammento LUBM

Prima prova del workflow con **un caso, un modello locale e tre chiamate Ollama**.
Domanda: `AssistantProfessor ⊑ Faculty`?

1. **Baseline:** due assiomi LUBM sostengono la conclusione.
2. **Ritiro del supporto:** una nuova versione del contesto rimuove un assioma necessario.
3. **Controllo:** un ramo indipendente della stessa conversazione iniziale aggiunge un assioma irrilevante.

Misuriamo la revisione dopo un cambiamento delle evidenze, non la correzione spontanea di un errore
su evidenze immutate. La risposta iniziale deve essere corretta per interpretare il test come
ritiro di una credenza precedentemente giustificata.

**Ambito:** micro-ontologia estratta dal file LUBM della repository, non ragionamento sull'intera
ontologia. Contiene solo inclusioni positive fra classi nominate. In questo frammento la
raggiungibilità è un oracolo completo per la query; non è un reasoner OWL generale.
`NOT_ENTAILED` significa “non segue dal contesto”, non “la sua negazione è vera”.

## Avvio

Avviare Ollama e selezionare il kernel Python dell'ambiente della repository. Da PowerShell:

```powershell
py -3.13 -m venv .venv
.\.venv\Scripts\python.exe -m pip install -r requirements-belief-revision.txt
.\.venv\Scripts\python.exe -m ipykernel install --user --name epistemic-pilot --display-name "Python (epistemic pilot)"
```

Aprire questo file in un editor Jupyter e usare **Run All**. Il notebook non scarica modelli.
`MODEL` è modificabile nella prossima cella. Tutti i risultati vengono salvati in una nuova
cartella `generated/belief_revision/`; ogni chiamata viene registrata subito, anche in caso di errore.

In [1]:
from pathlib import Path
from datetime import datetime, timezone
from collections import deque
import hashlib
import importlib.metadata
import json
import os
import platform
import time
import urllib.request
import urllib.error
import uuid

import rdflib
from rdflib import Namespace, RDFS
import jsonschema
from IPython.display import display, Markdown

MODEL = "qwen2.5-coder:7b"  # modello della fascia 4B–7B già disponibile in locale
OLLAMA_URL = os.environ.get("OLLAMA_HOST", "http://localhost:11434").rstrip("/")
if not OLLAMA_URL.startswith(("http://", "https://")):
    OLLAMA_URL = "http://" + OLLAMA_URL
OPTIONS = {"temperature": 0, "seed": 42, "num_ctx": 4096, "num_predict": 384}
TIMEOUT_S = 180

candidates = [Path.cwd(), Path.cwd() / "neurosymbolic-epistemic", Path.cwd().parent]
ROOT = next((p.resolve() for p in candidates if (p / "univ-bench-inst.ttl").exists()), None)
if ROOT is None:
    raise FileNotFoundError("Avviare il notebook dalla repository.")
SOURCE = ROOT / "univ-bench-inst.ttl"

def api(path, payload=None):
    data = None if payload is None else json.dumps(payload).encode("utf-8")
    request = urllib.request.Request(OLLAMA_URL + path, data=data,
                                     headers={"Content-Type": "application/json"})
    try:
        with urllib.request.urlopen(request, timeout=TIMEOUT_S) as response:
            return json.load(response)
    except (urllib.error.URLError, TimeoutError) as exc:
        raise RuntimeError(f"Ollama non raggiungibile o richiesta fallita: {exc}") from exc

installed = api("/api/tags")["models"]
model_info = next((m for m in installed if m["name"] == MODEL), None)
if model_info is None:
    raise ValueError(f"Modello {MODEL!r} assente. Disponibili: {[m['name'] for m in installed]}")
print("Modello:", MODEL, "| dimensione su disco:", round(model_info["size"] / 1e9, 3), "GB")
print("Digest:", model_info["digest"])

Modello: qwen2.5-coder:7b | dimensione su disco: 4.683 GB
Digest: dae161e27b0e90dd1856c8bb3209201fd6736d8eb66298e75ed87571486f4364


## Costruzione del caso e risultato atteso

Gli assiomi vengono controllati nel Turtle originale; non sono relazioni inventate.
Gli ID rimangono stabili fra versioni. Il modello vede solo lo snapshot corrente e la
conversazione precedente, mai le etichette dell'oracolo.

Nel ramo di ritiro il nuovo snapshot **sostituisce** quello precedente. Non chiediamo al modello
di cambiare risposta né gli diciamo quale risposta sia corretta.

In [2]:
LUBM = Namespace("http://swat.cse.lehigh.edu/onto/univ-bench.owl#")
source_graph = rdflib.Graph().parse(SOURCE, format="turtle")
AXIOMS = {
    "A1": ("AssistantProfessor", "Professor"),
    "A2": ("Professor", "Faculty"),
    "A3": ("Faculty", "Employee"),  # aggiunta irrilevante per la query
}
for aid, (sub, sup) in AXIOMS.items():
    assert (LUBM[sub], RDFS.subClassOf, LUBM[sup]) in source_graph, (aid, sub, sup)

QUERY = ("AssistantProfessor", "Faculty")
SCENARIOS = {
    "baseline": {k: AXIOMS[k] for k in ("A1", "A2")},
    "withdrawal": {k: AXIOMS[k] for k in ("A1",)},
    "irrelevant_control": dict(AXIOMS),
}

def proof_path(axioms, sub, sup):
    # Completo SOLO per inclusioni positive fra classi nominate ordinarie.
    queue = deque([(sub, [])])
    visited = {sub}
    while queue:
        node, path = queue.popleft()
        if node == sup:
            return path
        for aid, (left, right) in axioms.items():
            if left == node and right not in visited:
                visited.add(right)
                queue.append((right, path + [aid]))
    return None

def expected(axioms):
    return "ENTAILED" if proof_path(axioms, *QUERY) is not None else "NOT_ENTAILED"

def snapshot(axioms):
    return "\n".join(f"[{aid}] :{sub} rdfs:subClassOf :{sup} ."
                     for aid, (sub, sup) in axioms.items())

assert expected(SCENARIOS["baseline"]) == "ENTAILED"
assert expected(SCENARIOS["withdrawal"]) == "NOT_ENTAILED"
assert expected(SCENARIOS["irrelevant_control"]) == "ENTAILED"
for name, axioms in SCENARIOS.items():
    print(name, "=>", expected(axioms), "| prova:", proof_path(axioms, *QUERY))
    print(snapshot(axioms), "\n")

baseline => ENTAILED | prova: ['A1', 'A2']
[A1] :AssistantProfessor rdfs:subClassOf :Professor .
[A2] :Professor rdfs:subClassOf :Faculty . 

withdrawal => NOT_ENTAILED | prova: None
[A1] :AssistantProfessor rdfs:subClassOf :Professor . 

irrelevant_control => ENTAILED | prova: ['A1', 'A2']
[A1] :AssistantProfessor rdfs:subClassOf :Professor .
[A2] :Professor rdfs:subClassOf :Faculty .
[A3] :Faculty rdfs:subClassOf :Employee . 



## Protocollo e registrazione

Usiamo lo stesso schema JSON in tutti i turni. I vincoli di formato agevolano il parsing,
ma non garantiscono correttezza semantica. Nessun retry selettivo o riparazione automatica:
risposte invalide e troncate sono registrate come tali. I due aggiornamenti partono dalla
**stessa risposta baseline**, senza vedere il risultato dell'altro ramo.

Le chiamate usano l'[API chat di Ollama](https://docs.ollama.com/api/chat) e gli
[output strutturati](https://docs.ollama.com/capabilities/structured-outputs).

In [3]:
SCHEMA = {
    "type": "object",
    "properties": {
        "status": {"type": "string", "enum": ["ENTAILED", "NOT_ENTAILED"]},
        "axiom_ids": {"type": "array", "items": {"type": "string"}},
        "explanation": {"type": "string"},
    },
    "required": ["status", "axiom_ids", "explanation"],
    "additionalProperties": False,
}
INSTRUCTIONS = """Evaluate the query using only the CURRENT SNAPSHOT of an ontology fragment.
Each axiom [ID] says that every member of the first class belongs to the second class.
Subclass inclusions are transitive. Use no outside knowledge or unstated relations.
ENTAILED means the query follows from the current axioms.
NOT_ENTAILED means it does not follow; it does NOT mean its negation is true.
A new snapshot replaces the previous one completely. Old axioms not listed are no longer available.
For ENTAILED, cite only IDs of current axioms sufficient to derive the query.
For NOT_ENTAILED, return an empty axiom_ids list and briefly describe the missing support.
Give a short explanation, not an extended reasoning trace.
Return only JSON matching this schema:
""" + json.dumps(SCHEMA)

def user_message(axioms, update=False):
    prefix = ("This is the new current snapshot. It replaces the previous snapshot. Re-evaluate the same query."
              if update else "This is the initial current snapshot. Evaluate the query.")
    return {"role": "user", "content": INSTRUCTIONS + "\n\n" + prefix
            + "\nCURRENT SNAPSHOT:\n" + snapshot(axioms)
            + f"\nQUERY: Is :{QUERY[0]} a subclass of :{QUERY[1]}?"}

run_id = datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%SZ") + "_" + uuid.uuid4().hex[:8]
RUN_DIR = ROOT / "generated" / "belief_revision" / run_id
RUN_DIR.mkdir(parents=True, exist_ok=False)
manifest = {
    "run_id": run_id, "model": model_info, "ollama_version": api("/api/version"),
    "options": OPTIONS, "timeout_s": TIMEOUT_S, "schema": SCHEMA,
    "source": SOURCE.name, "source_sha256": hashlib.sha256(SOURCE.read_bytes()).hexdigest(),
    "scope": "LUBM positive named-subclass fragment; not full OWL",
    "query": QUERY, "snapshots": SCENARIOS,
    "oracle": {name: expected(a) for name, a in SCENARIOS.items()},
    "python": platform.python_version(), "platform": platform.platform(),
    "packages": {p: importlib.metadata.version(p) for p in ("rdflib", "jsonschema", "ipykernel")},
}
(RUN_DIR / "manifest.json").write_text(json.dumps(manifest, indent=2), encoding="utf-8")

def call_model(name, messages):
    payload = {"model": MODEL, "messages": messages, "stream": False,
               "format": SCHEMA, "options": OPTIONS}
    record = {"scenario": name, "request": payload, "response": None,
              "parsed": None, "error": None}
    start = time.perf_counter()
    try:
        response = api("/api/chat", payload)
        record["response"] = response  # conserva risposta grezza e statistiche Ollama
        if response.get("done_reason") == "length":
            raise ValueError("Risposta troncata per limite di token")
        parsed = json.loads(response["message"]["content"])
        jsonschema.validate(parsed, SCHEMA)
        record["parsed"] = parsed
    except Exception as exc:
        record["error"] = f"{type(exc).__name__}: {exc}"
    record["elapsed_s"] = round(time.perf_counter() - start, 3)
    with (RUN_DIR / "responses.jsonl").open("a", encoding="utf-8") as f:
        f.write(json.dumps(record, ensure_ascii=False) + "\n")
    print(name, json.dumps(record["parsed"], ensure_ascii=False) if record["parsed"] else record["error"])
    return record

print("Output:", RUN_DIR)

Output: C:\Users\manud\Desktop\neurosymbolic - epistemic\neurosymbolic-epistemic\generated\belief_revision\20260926T143945Z_d4111c12


In [4]:
initial_message = user_message(SCENARIOS["baseline"])
records = [call_model("baseline", [initial_message])]
baseline = records[0]
if baseline["parsed"] is not None:
    # La risposta autentica del modello resta nella storia: nessuna risposta gold inserita.
    previous_reply = {"role": "assistant", "content": baseline["response"]["message"]["content"]}
    for name in ("withdrawal", "irrelevant_control"):
        messages = [initial_message, previous_reply, user_message(SCENARIOS[name], update=True)]
        records.append(call_model(name, messages))
else:
    print("Baseline non valida: rami non eseguiti. Il run resta registrato, senza sostituire la risposta.")

baseline {"status": "ENTAILED", "axiom_ids": ["A1", "A2"], "explanation": "A1 and A2 together establish that AssistantProfessor is a subclass of Faculty through transitive subclass inclusion."}


withdrawal {"status": "NOT_ENTAILED", "axiom_ids": [], "explanation": "The current snapshot only provides information about the subclass relationship between AssistantProfessor and Professor, but it does not provide any information about the relationship between Professor and Faculty. Therefore, it is not possible to determine if AssistantProfessor is a subclass of Faculty based on the given axioms."}


irrelevant_control {"status": "ENTAILED", "axiom_ids": ["A1", "A2"], "explanation": "A1 and A2 together establish that AssistantProfessor is a subclass of Faculty through transitive subclass inclusion."}


## Valutazione automatica

Separiamo correttezza dello status, esistenza degli ID citati e sufficienza del supporto.
Accettiamo qualunque insieme di assiomi valido, non soltanto un percorso gold.
Per `NOT_ENTAILED`, una lista vuota rispetta il formato ma **non costituisce una prova negativa**:
la correttezza viene stabilita dall'oracolo sullo snapshot completo.
Il testo della spiegazione viene mostrato per ispezione e non riceve un punteggio semantico automatico.

In [5]:
def score(record):
    axioms = SCENARIOS[record["scenario"]]
    parsed = record["parsed"]
    row = {"scenario": record["scenario"], "expected": expected(axioms),
           "predicted": None, "valid_output": parsed is not None,
           "status_correct": False, "citations_current": None,
           "support_sufficient": None, "support_minimal": None,
           "stale_ids": [], "protocol_success": False}
    if parsed is None:
        return row
    ids = set(parsed["axiom_ids"])
    current = ids <= axioms.keys()
    cited = {aid: axioms[aid] for aid in ids if aid in axioms}
    row.update(predicted=parsed["status"], status_correct=parsed["status"] == row["expected"],
               citations_current=current, stale_ids=sorted(ids - axioms.keys()))
    if parsed["status"] == "ENTAILED":
        sufficient = current and proof_path(cited, *QUERY) is not None
        minimal = sufficient and all(proof_path({k: v for k, v in cited.items() if k != aid}, *QUERY)
                                      is None for aid in cited)
        row.update(support_sufficient=sufficient, support_minimal=minimal)
        row["protocol_success"] = row["status_correct"] and sufficient
    else:
        row["protocol_success"] = row["status_correct"] and not ids
    return row

scores = [score(r) for r in records]
by_name = {s["scenario"]: s for s in scores}
eligible = by_name["baseline"]["protocol_success"]
summary = {
    "baseline_justified": eligible,
    "withdrawal_success_given_justified_baseline":
        by_name.get("withdrawal", {}).get("protocol_success") if eligible else None,
    "control_stability_given_justified_baseline":
        by_name.get("irrelevant_control", {}).get("protocol_success") if eligible else None,
    "scores": scores,
}
(RUN_DIR / "evaluation.json").write_text(json.dumps(summary, indent=2), encoding="utf-8")

header = "| Condizione | Atteso | Risposta | Status corretto | Supporto sufficiente | ID obsoleti/ignoti |\n|---|---|---|---|---|---|"
rows = [f"| {s['scenario']} | {s['expected']} | {s['predicted']} | {s['status_correct']} | {s['support_sufficient']} | {s['stale_ids']} |"
        for s in scores]
display(Markdown(header + "\n" + "\n".join(rows)))
print(json.dumps({k: v for k, v in summary.items() if k != "scores"}, indent=2))
if not eligible:
    print("La baseline non è corretta e giustificata: non interpretiamo gli aggiornamenti come revisione riuscita di una credenza giustificata.")
print("Risultati salvati in", RUN_DIR)

| Condizione | Atteso | Risposta | Status corretto | Supporto sufficiente | ID obsoleti/ignoti |
|---|---|---|---|---|---|
| baseline | ENTAILED | ENTAILED | True | True | [] |
| withdrawal | NOT_ENTAILED | NOT_ENTAILED | True | None | [] |
| irrelevant_control | ENTAILED | ENTAILED | True | True | [] |

{
  "baseline_justified": true,
  "withdrawal_success_given_justified_baseline": true,
  "control_stability_given_justified_baseline": true
}
Risultati salvati in C:\Users\manud\Desktop\neurosymbolic - epistemic\neurosymbolic-epistemic\generated\belief_revision\20260926T143945Z_d4111c12


## Come leggere questa prima prova

- Baseline corretta con `A1, A2`, ritiro `NOT_ENTAILED`, controllo `ENTAILED`: workflow riuscito su questo episodio.
- Baseline corretta ma ritiro ancora `ENTAILED`: mancato aggiornamento; controllare l'eventuale citazione di `A2` ormai ritirato.
- Controllo che cambia a `NOT_ENTAILED`: revisione ingiustificata dopo una modifica irrilevante.
- Baseline errata o supporto invalido: il prerequisito del test non è soddisfatto; gli altri turni restano descrittivi.

`qwen2.5-coder:7b` è già disponibile in locale; è un modello specializzato sul codice,
quindi questa prova non rappresenta automaticamente modelli generalisti della stessa taglia.
Un episodio e una singola generazione non consentono stime di robustezza, calibrazione o conclusioni
sull'epistemic agency generale. Anche temperature zero e seed fisso non garantiscono identità fra hardware/versioni.
L'oracolo verifica il supporto dichiarato, non il processo interno del modello.

Il prossimo confronto utile è ripetere lo stesso episodio con un altro modello locale, poi variare
nomi delle classi, query e percorsi alternativi. Per passare all'intera LUBM occorre un reasoner OWL
completo e una nuova validazione dei risultati attesi dopo ciascuna modifica.

# Esperimento 2 — Ontologia LUBM completa

Questa sezione è autonoma: eseguire le celle da qui per evitare di ripetere il pilot precedente.
Invia **tutte le triple di `univ-bench-inst.ttl`**, comprese TBox, proprietà, restrizioni e individui.
Il Turtle è riserializzato in forma compatta, ma una verifica di isomorfismo garantisce che non
venga omessa alcuna tripla. Nel ritiro rimuoviamo esclusivamente le triple dichiarate nel protocollo.
Non eliminiamo tutte le occorrenze di una classe o un intero blocco testuale.

**Citazioni con URI:** una classe ha un URI; un assioma OWL in generale non ne possiede uno proprio.
Per queste query accettiamo triple `subject`, `predicate`, `object` con URI completi, che identificano
senza ambiguità le inclusioni esplicite usate. Non chiediamo nomi artificiali come `A1`.
È un formato sufficiente per i due casi scelti, non una rappresentazione generale di assiomi OWL
con restrizioni e blank node. Le spiegazioni in linguaggio naturale restano da ispezionare.

**Ground truth:** HermiT tramite Owlready2 classifica ogni snapshot completo in un World isolato,
controlla la consistenza e determina l'entailment. Anche gli assiomi citati vengono verificati
con HermiT. L'oracolo non viene mostrato al modello. Non riutilizziamo la BFS del pilot come
oracolo per l'intera OWL.

Installazione aggiuntiva (Java è necessario e deve essere disponibile nel PATH):
```powershell
.\.venv\Scripts\python.exe -m pip install -r requirements-belief-revision-full.txt
```
Kernel: **Python (epistemic pilot)**. Documentazione del reasoner:
[Owlready2 — Reasoning](https://owlready2.readthedocs.io/en/latest/reasoning.html).

In [1]:
from pathlib import Path
from datetime import datetime, timezone
import hashlib
import importlib.metadata
import io
import json
import os
import platform
import shutil
import subprocess
import time
import urllib.request
import urllib.error
import uuid

import rdflib
from rdflib import RDF, RDFS, OWL, URIRef, Namespace
from rdflib.compare import isomorphic, to_isomorphic
import jsonschema
import owlready2
from IPython.display import display, Markdown

FULL_MODEL = "qwen2.5-coder:7b"
# Due ontologie complete nella conversazione; 4096 token del pilot non bastano.
FULL_OPTIONS = {"temperature": 0, "seed": 42, "num_ctx": 32768, "num_predict": 1024}
FULL_TIMEOUT_S = 600
FULL_HOST = os.environ.get("OLLAMA_HOST", "http://localhost:11434").rstrip("/")
if not FULL_HOST.startswith(("http://", "https://")):
    FULL_HOST = "http://" + FULL_HOST
full_candidates = [Path.cwd(), Path.cwd() / "neurosymbolic-epistemic", Path.cwd().parent]
FULL_ROOT = next((p.resolve() for p in full_candidates if (p / "univ-bench-inst.ttl").exists()), None)
if FULL_ROOT is None:
    raise FileNotFoundError("Avviare dalla repository.")
FULL_SOURCE = FULL_ROOT / "univ-bench-inst.ttl"
FULL_NS = Namespace("http://swat.cse.lehigh.edu/onto/univ-bench.owl#")
java_path = shutil.which("java")
if java_path is None:
    raise RuntimeError("Java richiesto per HermiT: installare un JDK e renderlo disponibile nel PATH.")
owlready2.JAVA_EXE = java_path

def full_api(path, payload=None):
    data = None if payload is None else json.dumps(payload).encode("utf-8")
    request = urllib.request.Request(FULL_HOST + path, data=data,
                                     headers={"Content-Type": "application/json"})
    try:
        with urllib.request.urlopen(request, timeout=FULL_TIMEOUT_S) as response:
            return json.load(response)
    except urllib.error.HTTPError as exc:
        detail = exc.read().decode("utf-8", errors="replace")
        raise RuntimeError(f"Ollama HTTP {exc.code}: {detail}") from exc

full_models = full_api("/api/tags")["models"]
full_model_info = next((m for m in full_models if m["name"] == FULL_MODEL), None)
if full_model_info is None:
    raise ValueError(f"Modello non installato: {FULL_MODEL}")
full_show = full_api("/api/show", {"model": FULL_MODEL})
native_context = [v for k, v in full_show.get("model_info", {}).items() if k.endswith(".context_length")]
if native_context and min(native_context) < FULL_OPTIONS["num_ctx"]:
    raise ValueError("La finestra richiesta supera quella dichiarata dal modello.")

full_run_id = datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%SZ") + "_" + uuid.uuid4().hex[:8]
FULL_RUN_DIR = FULL_ROOT / "generated" / "belief_revision_full" / full_run_id
FULL_RUN_DIR.mkdir(parents=True, exist_ok=False)
full_source_graph = rdflib.Graph().parse(FULL_SOURCE, format="turtle")
if list(full_source_graph.objects(None, OWL.imports)):
    raise ValueError("Import OWL esterni: materializzarli prima di garantire contesto completo.")
print("Modello:", FULL_MODEL, "| triple nel file:", len(full_source_graph))
print("Contesto richiesto:", FULL_OPTIONS["num_ctx"], "token | output:", FULL_RUN_DIR)

Modello: qwen2.5-coder:7b | triple nel file: 319
Contesto richiesto: 32768 token | output: C:\Users\manud\Desktop\neurosymbolic - epistemic\neurosymbolic-epistemic\generated\belief_revision_full\20260926T153409Z_49706adb


## Due casi e sei chiamate

**Caso completo semplice:** stessa query del pilot, `AssistantProfessor ⊑ Faculty`.
Baseline con tutta LUBM; ritiro di `Professor ⊑ Faculty`; controllo con una nuova annotazione
irrilevante. I due aggiornamenti ripartono separatamente dalla risposta baseline.

**Caso più difficile:** query `AssistantProfessor ⊑ Employee`. La LUBM originale contiene
la catena di tre inclusioni `AssistantProfessor → Professor → Faculty → Employee`.
Aggiungiamo un assioma sintetico, esplicitamente documentato: `AssistantProfessor ⊑ Faculty`.
Esistono quindi due supporti. Nei due rami indipendenti:

- ritiriamo `Professor ⊑ Faculty`: la via alternativa mantiene la conclusione;
- ritiriamo anche `AssistantProfessor ⊑ Faculty`: la conclusione non segue più.

Il secondo caso è un test di ridondanza delle evidenze: la maggiore difficoltà progettata
consiste nel decidere se una rimozione richiede davvero di cambiare risposta. Non è una
difficoltà empiricamente già dimostrata; una strategia efficace può usare la via più corta.

In [2]:
def full_copy(graph):
    result = rdflib.Graph()
    for prefix, namespace in graph.namespaces():
        result.bind(prefix, namespace)
    for triple in graph:
        result.add(triple)
    return result

bridge = (FULL_NS.Professor, RDFS.subClassOf, FULL_NS.Faculty)
shortcut = (FULL_NS.AssistantProfessor, RDFS.subClassOf, FULL_NS.Faculty)
assert bridge in full_source_graph and shortcut not in full_source_graph
simple_removed = full_copy(full_source_graph)
simple_removed.remove(bridge)
simple_control = full_copy(full_source_graph)
control_annotation = (FULL_NS.University, RDFS.comment, rdflib.Literal("Irrelevant annotation for this experiment."))
simple_control.add(control_annotation)
hard_baseline = full_copy(full_source_graph)
hard_baseline.add(shortcut)
hard_partial = full_copy(hard_baseline)
hard_partial.remove(bridge)
hard_removed = full_copy(hard_partial)
hard_removed.remove(shortcut)

FULL_CASES = {
    "full_simple": {
        "query": (str(FULL_NS.AssistantProfessor), str(FULL_NS.Faculty)),
        "graphs": {"baseline": full_source_graph, "withdrawal": simple_removed,
                   "irrelevant_control": simple_control},
        "expected_design": {"baseline": "ENTAILED", "withdrawal": "NOT_ENTAILED", "irrelevant_control": "ENTAILED"},
    },
    "redundant_support": {
        "query": (str(FULL_NS.AssistantProfessor), str(FULL_NS.Employee)),
        "graphs": {"baseline": hard_baseline, "withdraw_one_path": hard_partial,
                   "withdraw_both_paths": hard_removed},
        "expected_design": {"baseline": "ENTAILED", "withdraw_one_path": "ENTAILED", "withdraw_both_paths": "NOT_ENTAILED"},
    },
}

# Controlli sulle modifiche: preserviamo ogni altra tripla, inclusi i blank node.
assert set(full_source_graph) - set(simple_removed) == {bridge}
assert not set(simple_removed) - set(full_source_graph)
assert set(simple_control) - set(full_source_graph) == {control_annotation}
assert set(hard_baseline) - set(hard_partial) == {bridge}
assert set(hard_baseline) - set(hard_removed) == {bridge, shortcut}

full_oracle_cache = {}
def full_entails(graph, query):
    # World nuovo a ogni classificazione: nessuna inferenza del turno precedente sopravvive.
    key = (to_isomorphic(graph).graph_digest(), tuple(query))
    if key in full_oracle_cache:
        return full_oracle_cache[key]
    world = owlready2.World()
    try:
        world.get_ontology("http://swat.cse.lehigh.edu/onto/univ-bench.owl").load(
            fileobj=io.BytesIO(graph.serialize(format="xml", encoding="utf-8")))
        sub, sup = (world[uri] for uri in query)
        if sub is None or sup is None:
            raise ValueError("Classi della query non dichiarate nel grafo dell'oracolo.")
        owlready2.sync_reasoner(world, debug=0)
        ancestors = sub.ancestors()
        if owlready2.Nothing in ancestors:
            raise ValueError("Classe soggetto insoddisfacibile: entailment vacuo escluso dal pilot.")
        result = "ENTAILED" if sup in ancestors else "NOT_ENTAILED"
        full_oracle_cache[key] = result
        return result
    finally:
        world.close()

full_snapshots, full_gold = {}, {}
for case_name, case in FULL_CASES.items():
    full_snapshots[case_name], full_gold[case_name] = {}, {}
    for condition, graph in case["graphs"].items():
        turtle = graph.serialize(format="turtle")
        assert isomorphic(graph, rdflib.Graph().parse(data=turtle, format="turtle"))
        full_snapshots[case_name][condition] = turtle
        full_gold[case_name][condition] = full_entails(graph, case["query"])
        assert full_gold[case_name][condition] == case["expected_design"][condition]
        (FULL_RUN_DIR / f"{case_name}_{condition}.ttl").write_text(turtle, encoding="utf-8")
        print(case_name, condition, "| triple:", len(graph), "| HermiT:", full_gold[case_name][condition])

full_simple baseline | triple: 319 | HermiT: ENTAILED


full_simple withdrawal | triple: 318 | HermiT: NOT_ENTAILED


full_simple irrelevant_control | triple: 320 | HermiT: ENTAILED


redundant_support baseline | triple: 320 | HermiT: ENTAILED


redundant_support withdraw_one_path | triple: 319 | HermiT: ENTAILED


redundant_support withdraw_both_paths | triple: 318 | HermiT: NOT_ENTAILED


## Prompt e gestione della finestra di contesto

Il modello riceve istruzioni, schema della risposta, intera ontologia e query. Ogni aggiornamento
include la richiesta iniziale completa, la risposta autentica e la nuova ontologia completa.
Le evidenze ritirate rimangono nella cronologia ma non valgono nello snapshot corrente.
Le etichette delle condizioni, il tipo di modifica e i risultati attesi non sono inviati.

La finestra viene portata a 32768 token. Un limite conservativo basato sui byte UTF-8, con margine
per template e generazione, blocca richieste troppo grandi; non tagliamo mai l'ontologia per farla
entrare. Registriamo inoltre `prompt_eval_count` e la finestra effettiva riportata da Ollama.
Il limite in byte è intenzionalmente prudente per questo modello a tokenizzazione byte-level;
se si cambia famiglia di modello occorre rivalutare il controllo della tokenizzazione.

In [3]:
FULL_SCHEMA = {
    "type": "object",
    "properties": {
        "status": {"type": "string", "enum": ["ENTAILED", "NOT_ENTAILED"]},
        "axioms": {"type": "array", "items": {
            "type": "object",
            "properties": {
                "subject": {"type": "string", "pattern": r"^https?://[A-Za-z0-9_:/#.-]+$"},
                "predicate": {"type": "string", "const": str(RDFS.subClassOf)},
                "object": {"type": "string", "pattern": r"^https?://[A-Za-z0-9_:/#.-]+$"}},
            "required": ["subject", "predicate", "object"], "additionalProperties": False}},
        "explanation": {"type": "string"},
    },
    "required": ["status", "axioms", "explanation"], "additionalProperties": False,
}
FULL_INSTRUCTIONS = """Evaluate the query under OWL semantics using only the CURRENT ONTOLOGY.
The full ontology is provided in Turtle, including its individuals and restrictions.
ENTAILED means the query follows; NOT_ENTAILED means it does not follow, not that its negation is true.
A new current ontology replaces the previous ontology completely. Old axioms absent from it no longer apply.
Use no outside domain knowledge. Multiple derivations may exist; check the current ontology.
For ENTAILED, report asserted named-class rdfs:subClassOf triples sufficient to support the query.
For the queries in this experiment this citation format suffices whenever the answer is entailed.
Use complete absolute URIs in subject, predicate, object; no prefixes, blank nodes, or artificial axiom IDs.
Do not cite a derived triple as if it were asserted. Avoid unnecessary triples.
For NOT_ENTAILED, return an empty axioms array and briefly explain the missing support.
Provide a concise explanation and return only JSON matching the schema:
""" + json.dumps(FULL_SCHEMA)

def full_message(case_name, condition):
    query = FULL_CASES[case_name]["query"]
    update = ("This is the initial ontology. Evaluate the query." if condition == "baseline" else
              "This is the new current ontology. It replaces the previous ontology. Re-evaluate the same query.")
    return {"role": "user", "content": FULL_INSTRUCTIONS + "\n" + update
            + "\nCURRENT ONTOLOGY (Turtle):\n" + full_snapshots[case_name][condition]
            + f"\nQUERY: Is <{query[0]}> a subclass of <{query[1]}>?"}

full_manifest = {
    "run_id": full_run_id, "model": full_model_info, "model_details": full_show,
    "ollama_version": full_api("/api/version"), "options": FULL_OPTIONS,
    "source": FULL_SOURCE.name, "source_sha256": hashlib.sha256(FULL_SOURCE.read_bytes()).hexdigest(),
    "oracle": "HermiT through Owlready2; isolated World per snapshot",
    "packages": {p: importlib.metadata.version(p) for p in ("owlready2", "rdflib", "jsonschema")},
    "java": subprocess.run([java_path, "-version"], capture_output=True, text=True).stderr,
    "python": platform.python_version(), "platform": platform.platform(),
    "schema": FULL_SCHEMA, "gold": full_gold,
    "synthetic_shortcut": [str(x) for x in shortcut],
    "removed_bridge": [str(x) for x in bridge],
    "snapshot_sha256": {cn: {condition: hashlib.sha256(t.encode()).hexdigest() for condition,t in snapshots.items()}
                        for cn,snapshots in full_snapshots.items()},
    "queries": {name: case["query"] for name, case in FULL_CASES.items()},
}
(FULL_RUN_DIR / "manifest.json").write_text(json.dumps(full_manifest, indent=2), encoding="utf-8")

def full_call(case_name, condition, messages):
    payload = {"model": FULL_MODEL, "messages": messages, "format": FULL_SCHEMA,
               "stream": False, "options": FULL_OPTIONS}
    record = {"case": case_name, "condition": condition, "request": payload,
              "response": None, "parsed": None, "error": None}
    start = time.perf_counter()
    try:
        budget = len(json.dumps(messages, ensure_ascii=False).encode("utf-8")) + len(json.dumps(FULL_SCHEMA).encode()) + 4096 + FULL_OPTIONS["num_predict"]
        record["conservative_context_budget"] = budget
        if budget > FULL_OPTIONS["num_ctx"]:
            raise ValueError(f"Contesto potenzialmente eccessivo: {budget}; nessuna richiesta inviata.")
        response = full_api("/api/chat", payload)
        record["response"] = response
        record["ollama_ps"] = full_api("/api/ps")
        loaded = next((m for m in record["ollama_ps"].get("models", []) if m.get("digest") == full_model_info["digest"]), {})
        actual_ctx = loaded.get("context_length")
        if actual_ctx is not None and actual_ctx < budget:
            raise ValueError(f"Contesto effettivo insufficiente: {actual_ctx} < {budget}")
        if response.get("prompt_eval_count", 0) + FULL_OPTIONS["num_predict"] >= FULL_OPTIONS["num_ctx"]:
            raise ValueError("Prompt troppo vicino al limite: verificare possibile troncamento.")
        if response.get("done_reason") == "length":
            raise ValueError("Output troncato: conservato, ma non valutabile.")
        parsed = json.loads(response["message"]["content"])
        jsonschema.validate(parsed, FULL_SCHEMA)
        record["parsed"] = parsed
    except Exception as exc:
        record["error"] = f"{type(exc).__name__}: {exc}"
    record["elapsed_s"] = round(time.perf_counter() - start, 3)
    with (FULL_RUN_DIR / "responses.jsonl").open("a", encoding="utf-8") as f:
        f.write(json.dumps(record, ensure_ascii=False) + "\n")
    print(case_name, condition, "|", record["elapsed_s"], "s |",
          json.dumps(record["parsed"], ensure_ascii=False) if record["parsed"] else record["error"], flush=True)
    return record

In [4]:
full_records = []
for case_name, case in FULL_CASES.items():
    initial = full_message(case_name, "baseline")
    first = full_call(case_name, "baseline", [initial])
    full_records.append(first)
    if first["parsed"] is None:
        print("Baseline non valida: i rami di questo caso non vengono eseguiti.")
        continue
    authentic_reply = {"role": "assistant", "content": first["response"]["message"]["content"]}
    for condition in case["graphs"]:
        if condition != "baseline":
            full_records.append(full_call(case_name, condition,
                [initial, authentic_reply, full_message(case_name, condition)]))

full_simple baseline | 37.461 s | {"status": "ENTAILED", "axioms": [{"subject": "http://swat.cse.lehigh.edu/onto/univ-bench.owl#AssistantProfessor", "predicate": "http://www.w3.org/2000/01/rdf-schema#subClassOf", "object": "http://swat.cse.lehigh.edu/onto/univ-bench.owl#Professor"}, {"subject": "http://swat.cse.lehigh.edu/onto/univ-bench.owl#Professor", "predicate": "http://www.w3.org/2000/01/rdf-schema#subClassOf", "object": "http://swat.cse.lehigh.edu/onto/univ-bench.owl#Faculty"}], "explanation": "The query follows from the asserted subclass relationships in the ontology."}


full_simple withdrawal | 59.02 s | {"status": "NOT_ENTAILED", "axioms": [], "explanation": "The current ontology does not assert that <http://swat.cse.lehigh.edu/onto/univ-bench.owl#AssistantProfessor> is a subclass of <http://swat.cse.lehigh.edu/onto/univ-bench.owl#Faculty>. The only subclass relationship provided is that <http://swat.cse.lehigh.edu/onto/univ-bench.owl#AssistantProfessor> is a subclass of <http://swat.cse.lehigh.edu/onto/univ-bench.owl#Professor>, and <http://swat.cse.lehigh.edu/onto/univ-bench.owl#Professor> is a subclass of <http://swat.cse.lehigh.edu/onto/univ-bench.owl#Faculty>. However, this does not directly entail that <http://swat.cse.lehigh.edu/onto/univ-bench.owl#AssistantProfessor> is a subclass of <http://swat.cse.lehigh.edu/onto/univ-bench.owl#Faculty>."}


full_simple irrelevant_control | 12.228 s | {"status": "NOT_ENTAILED", "axioms": [], "explanation": "The current ontology does not include any subclass relationships between AssistantProfessor and Faculty."}


redundant_support baseline | 16.727 s | {"status": "NOT_ENTAILED", "axioms": [], "explanation": "The current ontology does not assert that AssistantProfessor is a subclass of Employee. The only subclass relationship involving Employee is that AdministrativeStaff is a subclass of Employee, and AssistantProfessor is not listed as a subclass of Employee."}


redundant_support withdraw_one_path | 21.579 s | {"status": "NOT_ENTAILED", "axioms": [], "explanation": "The current ontology does not assert that AssistantProfessor is a subclass of Employee. The only subclass relationship involving Employee is that AdministrativeStaff is a subclass of Employee, and AssistantProfessor is not listed as a subclass of Employee."}


redundant_support withdraw_both_paths | 21.011 s | {"status": "NOT_ENTAILED", "axioms": [], "explanation": "The current ontology does not assert that AssistantProfessor is a subclass of Employee. The only subclass relationship involving Employee is that AdministrativeStaff is a subclass of Employee, and AssistantProfessor is not listed as a subclass of Employee."}


## Valutazione delle risposte e delle citazioni URI

Controlliamo che le triple citate siano inclusioni fra classi nominate, appartengano allo
snapshot corrente e siano sufficienti, insieme alla semantica OWL, per derivare la query.
L'ontologia completa determina il risultato atteso; il solo sottoinsieme citato determina
la sufficienza della giustificazione. Prove alternative sono ammesse. La minimalità è
per inclusione, non necessariamente per cardinalità, ed è separata dalla correttezza.

Per `NOT_ENTAILED`, nessuna lista di citazioni vuota dimostra da sola il non-entailment:
lo certifica il controllo HermiT sullo snapshot completo. Errori del reasoner interrompono
la valutazione, senza essere trasformati in risposte negative.

In [5]:
def full_support_graph(triples, query):
    graph = rdflib.Graph()
    for uri in query:
        graph.add((URIRef(uri), RDF.type, OWL.Class))
    for sub, pred, sup in triples:
        graph.add((sub, pred, sup))
        graph.add((sub, RDF.type, OWL.Class))
        graph.add((sup, RDF.type, OWL.Class))
    return graph

def full_score(record):
    name, condition = record["case"], record["condition"]
    graph = FULL_CASES[name]["graphs"][condition]
    query = FULL_CASES[name]["query"]
    parsed = record["parsed"]
    row = {"case": name, "condition": condition, "gold": full_gold[name][condition],
           "predicted": None, "valid_output": parsed is not None, "status_correct": False,
           "citations_current": None, "support_sufficient": None, "support_minimal": None,
           "invalid_or_absent_triples": [], "protocol_success": False, "elapsed_s": record["elapsed_s"]}
    if parsed is None:
        return row
    triples = set()
    invalid = []
    for a in parsed["axioms"]:
        values = [a[k] for k in ("subject", "predicate", "object")]
        triple = tuple(URIRef(v) for v in values)
        allowed = (all(v.startswith(("http://", "https://")) and not any(c.isspace() for c in v) for v in values)
                   and triple[1] == RDFS.subClassOf
                   and (triple[0], RDF.type, OWL.Class) in graph
                   and (triple[2], RDF.type, OWL.Class) in graph)
        if not allowed or triple not in graph:
            invalid.append(a)
        else:
            triples.add(triple)
    row.update(predicted=parsed["status"], status_correct=parsed["status"] == row["gold"],
               citations_current=not invalid, invalid_or_absent_triples=invalid)
    if parsed["status"] == "ENTAILED":
        sufficient = not invalid and full_entails(full_support_graph(triples, query), query) == "ENTAILED"
        minimal = sufficient and all(full_entails(full_support_graph(triples - {t}, query), query) == "NOT_ENTAILED"
                                      for t in triples)
        row.update(support_sufficient=sufficient, support_minimal=minimal,
                   protocol_success=row["status_correct"] and sufficient)
    else:
        row["protocol_success"] = row["status_correct"] and not parsed["axioms"]
    return row

full_scores = [full_score(r) for r in full_records]
full_summary = {}
for name in FULL_CASES:
    group = {s["condition"]: s for s in full_scores if s["case"] == name}
    eligible = group["baseline"]["protocol_success"]
    full_summary[name] = {"baseline_justified": eligible,
        "branches_given_justified_baseline": {c: group.get(c, {}).get("protocol_success") if eligible else None
                                              for c in FULL_CASES[name]["graphs"] if c != "baseline"}}
(FULL_RUN_DIR / "evaluation.json").write_text(json.dumps({"summary": full_summary, "scores": full_scores}, indent=2), encoding="utf-8")
table = "| Caso | Condizione | HermiT | LLM | Supporto sufficiente | Esito protocollo |\n|---|---|---|---|---|---|"
for s in full_scores:
    table += f"\n| {s['case']} | {s['condition']} | {s['gold']} | {s['predicted']} | {s['support_sufficient']} | {s['protocol_success']} |"
display(Markdown(table))
print(json.dumps(full_summary, indent=2))
print("Risposte, snapshot completi, configurazione e valutazioni:", FULL_RUN_DIR)

| Caso | Condizione | HermiT | LLM | Supporto sufficiente | Esito protocollo |
|---|---|---|---|---|---|
| full_simple | baseline | ENTAILED | ENTAILED | True | True |
| full_simple | withdrawal | NOT_ENTAILED | NOT_ENTAILED | None | True |
| full_simple | irrelevant_control | ENTAILED | NOT_ENTAILED | None | False |
| redundant_support | baseline | ENTAILED | NOT_ENTAILED | None | False |
| redundant_support | withdraw_one_path | ENTAILED | NOT_ENTAILED | None | False |
| redundant_support | withdraw_both_paths | NOT_ENTAILED | NOT_ENTAILED | None | True |

{
  "full_simple": {
    "baseline_justified": true,
    "branches_given_justified_baseline": {
      "withdrawal": true,
      "irrelevant_control": false
    }
  },
  "redundant_support": {
    "baseline_justified": false,
    "branches_given_justified_baseline": {
      "withdraw_one_path": null,
      "withdraw_both_paths": null
    }
  }
}
Risposte, snapshot completi, configurazione e valutazioni: C:\Users\manud\Desktop\neurosymbolic - epistemic\neurosymbolic-epistemic\generated\belief_revision_full\20260926T153409Z_49706adb


## Interpretazione e limiti del secondo esperimento

Un buon esito richiede una baseline giustificata e aggiornamenti corretti. Nel caso con
ridondanza, mantenere la conclusione dopo il primo ritiro è una risposta corretta, non inerzia.
L'esito distingue quindi sensibilità alle evidenze e semplice propensione a cambiare risposta.

Le due query riguardano sottoclassi; il contesto comprende anche costrutti OWL più ricchi e ABox,
ma non stiamo ancora misurando sistematicamente il ragionamento su quei costrutti. La via alternativa
è sintetica e documentata, mentre tutti gli altri assiomi derivano dal file della repository.
L'oracolo assume che il contesto OWL fornito sia la fonte normativa, non certifica verità nel mondo.
Sei risposte e un solo modello restano una prova del workflow, non un benchmark statistico.

## Lettura del run eseguito

Run `20260926T153409Z_49706adb`, modello `qwen2.5-coder:7b`, sei risposte valide nel formato URI.
Questa nota riguarda soltanto il run indicato; non si aggiorna automaticamente rieseguendo le celle.

- **Caso completo semplice:** baseline corretta con supporto sufficiente; status corretto dopo il ritiro;
  risposta errata nel controllo irrilevante (`NOT_ENTAILED` invece di `ENTAILED`).
- **Ispezione manuale della spiegazione del ritiro:** pur rispondendo `NOT_ENTAILED`, il modello
  menziona ancora `Professor ⊑ Faculty` come relazione disponibile e sostiene che i due passaggi
  non implichino la conclusione. La spiegazione è incoerente con snapshot e transitività.
  Il successo automatico su questo turno riguarda status e formato delle citazioni, non la spiegazione.
- **Supporti alternativi:** baseline errata; il modello continua a rispondere `NOT_ENTAILED` nei
  due rami. Il ritiro di entrambe le vie ha l'etichetta corretta, ma non è evidenza di revisione
  riuscita perché la baseline non era giustificata.

I run precedenti restano su disco: `20260926T152829Z_5ee72298` usava uno schema più permissivo
che lasciava passare il predicato abbreviato; `20260926T153202Z_c4dcb8b0` ha incontrato errori
infrastrutturali dovuti a una regex non supportata dal parser della versione locale di Ollama.
Il run finale usa un pattern compatibile, verificato prima dell'esecuzione. Non abbiamo modificato
le query o i risultati attesi per ottenere risposte corrette.

Non attribuiamo causalmente gli errori alla sola lunghezza del contesto: rispetto al primo pilot
è cambiato anche il formato di output. Per isolare l'effetto della lunghezza servirà un confronto
snippet/intera ontologia a parità di schema URI e con più esempi.
